<a href="https://colab.research.google.com/github/sowrin-paul/flyrank-intern/blob/main/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

Task Type: ranking/scoring, with a binary classification model underneath it. The decision is "which pages does a reviewer open first", needs an ordered queue, not a single yes/no. The reference pipeline builds this by training a classifier (random_forest, chosen over logistic_regression and decision_tree) whose predicted probability becomes the rank-ordering score, then layers rule-based reason codes and a 5-way action label (monitor, refresh, refresh_and_review_ctr, refresh_and_review_engagement, expand_and_refresh) on top for explainability.

In [1]:
import pandas as pd

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

The real target used by the pipeline is is_declining_label, defined in 01_prepare_features.py as tred_direction == 'down' which is a single, simple, observed condition, not something I need to hand-construct. On the 30,000-row starter trend_direction/trend_pct can never be used as features for exactly this reason. They're only safe as the label, and even then only as a starting proxy for "this page is worth looking", to be replaced with a real forward-window outcome once more than 90days of history is available.

In [6]:
df = pd.read_csv("/content/content_refresh_anonymized.csv")
df = df[(df.impressions_90d > 0) & (df.content_age_days >= 90)].drop_duplicates("content_id").copy()

df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
# df["is_declining_lable"].value_counts()

print(f"Rows: {len(df):,}")
print(f"Declining rate: {df['is_declining_label'].mean():.3f}")

Rows: 30,000
Declining rate: 0.542


## 3. Success metric

*One metric you can defend. What number means 'good'?*

Precision@50 - this is the exact metric the reference pipeline optimizes model selection on (Best model: random_forest selected by precision_at_50). A reviewer can only get through so many pages, so precision at the top of the queue is what matters, not overall accuracy.

| Model | Precision@50 | ROC AUC |
|---|---:|---:|
| baseline_rules | 0.240 | 0.627 |
| logistic_regression | 0.400 | 0.700 |
| decision_tree | 0.540 | 0.742 |
| **random_forest** | **0.704** | 0.750 |

The random forest beats the hand-written rule baseline by roughly **3.08x** at precision@50 (0.240 to 0.740) - the single clearest number for why build a model at all.

In [7]:
# No code

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [11]:
queue = pd.read_csv("/content/refresh_queue_sample.csv")
cols = ["final_rank", "content_id", "client_id", "final_refresh_score", "best_model_probability", "confidence", "suggested_action", "is_declining_label", "impressions_90d", "avg_position", "ctr", "trend_direction"]

queue[cols].head(5)


,final_rank,content_id,client_id,final_refresh_score,best_model_probability,confidence,suggested_action,is_declining_label,impressions_90d,avg_position,ctr,trend_direction
0,1,content_1f080331fa2b,client_3fdba35f04,81.636697,0.782079,high,refresh_and_review_ctr,1,12834,6.8,0.05,down
1,2,content_6aa43079fb0c,client_3fdba35f04,81.447656,0.788105,high,refresh_and_review_ctr,1,8064,3.8,0.07,down
2,3,content_d6570c51c9bd,client_3fdba35f04,81.430346,0.847372,medium,refresh_and_review_ctr,1,2498,10.1,0.00,down
3,4,content_72e800a9c214,client_3fdba35f04,81.034960,0.774371,high,refresh_and_review_ctr,1,13790,8.2,0.12,down
4,5,content_e04eb9549989,client_3fdba35f04,80.873188,0.814805,medium,refresh_and_review_ctr,1,3393,3.6,0.09,down


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

The pipeline's own feature importances make this concrete: the two strongest predictors are `days_with_impressions` (0.158) and `log_impressions_90d` (0.128) — meaning *how consistently* a page gets seen matters more than any single threshold on any one day. A fixed rule built from a handful of if/else thresholds (the `baseline_rules` row above) can only check a few conditions at fixed cutoffs, and it caps out at Precision@50 = 0.240. The random forest, weighing 10+ signals (`avg_position`, `content_age_days`, `char_count`, `ctr`, `scroll_rate`, etc.) together and letting their *combinations* matter, reaches 0.740 — a ~3x improvement at the exact point that matters for a reviewer's limited weekly capacity.

The output also isn't one bucket. Across the **full** scored queue (per `model_report.md`), items split into 5 distinct actions: `monitor` (13,093), `refresh` (8,178), `refresh_and_review_ctr` (6,657), `refresh_and_review_engagement` (1,990), `expand_and_refresh` (82) — a pattern a single rule would flatten into "flag or don't," losing the information about *what kind* of fix a page needs. The code cell below spot-checks this on the published top-200 sample: as expected, only the higher-severity actions (`refresh`, `refresh_and_review_ctr`, `refresh_and_review_engagement`) appear, since `monitor` and `expand_and_refresh` are lower-priority outcomes that don't rank into the top 200.

In [12]:
action_counts = queue["suggested_action"].value_counts()
print(action_counts)

suggested_action
refresh_and_review_ctr           130
refresh                           35
refresh_and_review_engagement     35
Name: count, dtype: int64


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.